In [3]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
from pathlib import Path
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report
from datetime import datetime

In [ ]:
# CONFIG
MODEL_PATH = Path("models/best_model.pkl")
X_TEST_CSV = Path("data/processed/X_test.csv")
Y_TEST_CSV = Path("data/processed/y_test.csv")
OUT_DIR = Path("reports/interpretability")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
N_FEATURES_TO_SHOW = 15       # top features to display
N_LIME_FEATURES = 10          # features in LIME local explanation
RANDOM_STATE = 42


In [7]:
# LOAD ARTIFACTS

N_FEATURES_TO_SHOW = 15       # top features to display
N_LIME_FEATURES = 10          # features in LIME local explanation
RANDOM_STATE = 42

In [10]:
print("Loading model & test data…")
pipe = joblib.load(MODEL_PATH)              # sklearn Pipeline(preprocess, model)
X_test = pd.read_csv(X_TEST_CSV)
y_test = pd.read_csv(Y_TEST_CSV).squeeze()

Loading model & test data…


FileNotFoundError: [Errno 2] No such file or directory: 'models/best_model.pkl'

In [ ]:
# -----------------------------
# LOAD ARTIFACTS
# -----------------------------

# Keep original (raw) feature names before transformation
raw_feature_names = list(X_test.columns)

# Predict once (sanity check + metrics)
print("Evaluating model…")
y_proba = pipe.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)
report = classification_report(y_test, y_pred, output_dict=True)
pd.DataFrame(report).to_csv(OUT_DIR / "classification_report.csv")
print(pd.DataFrame(report).round(3))

In [ ]:
# ------------------------------------------------------------
# 1) FEATURE IMPORTANCE
#    a) Model .feature_importances_ when available (tree models)
#    b) Permutation importance (model-agnostic, works for any model)
# ------------------------------------------------------------
def get_model_from_pipeline(pipeline):
    # assumes last step is the estimator
    return pipeline[-1]

def get_preprocessor_from_pipeline(pipeline):
    # assumes first step is the preprocessor (ColumnTransformer or similar)
    # fallback: None
    try:
        return pipeline[0]
    except Exception:
        return None

def transformed_feature_names(preprocessor, raw_cols):
    """
    Best-effort recovery of transformed feature names from ColumnTransformer.
    If unavailable, fall back to raw feature names length.
    """
    try:
        # For sklearn >=1.0 transformers usually expose get_feature_names_out
        names = preprocessor.get_feature_names_out(raw_cols)
        return list(names)
    except Exception:
        # Fallback: return placeholders
        return [f"feat_{i}" for i in range(preprocessor.transform(pd.DataFrame(columns=raw_cols)).shape[1])]

estimator = get_model_from_pipeline(pipe)
preproc = get_preprocessor_from_pipeline(pipe)

if preproc is not None:
    try:
        X_trans = preproc.transform(X_test)
        feature_names_trans = transformed_feature_names(preproc, raw_feature_names)
    except Exception:
        # some preprocessors need fit; pipeline should already be fit though
        X_trans = pipe[:-1].transform(X_test)
        feature_names_trans = [f"feat_{i}" for i in range(X_trans.shape[1])]
else:
    X_trans = X_test.values
    feature_names_trans = raw_feature_names

# a) Tree-based feature_importances_
tree_importance_df = None
if hasattr(estimator, "feature_importances_"):
    fi = estimator.feature_importances_
    tree_importance_df = (
        pd.DataFrame({"feature": feature_names_trans, "importance": fi})
        .sort_values("importance", ascending=False)
        .head(N_FEATURES_TO_SHOW)
    )
    tree_importance_df.to_csv(OUT_DIR / "feature_importance_tree.csv", index=False)

    plt.figure()
    tree_importance_df[::-1].plot(
        x="feature", y="importance", kind="barh", legend=False
    )
    plt.title("Feature Importance (Model-based)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "feature_importance_tree.png", dpi=200)
    plt.close()

# b) Permutation importance (works for any estimator)
print("Computing permutation importance (this may take a moment)…")
perm = permutation_importance(
    pipe, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, scoring="f1"
)
perm_df = (
    pd.DataFrame(
        {"feature": raw_feature_names, "importance_mean": perm.importances_mean, "importance_std": perm.importances_std}
    )
    .sort_values("importance_mean", ascending=False)
    .head(N_FEATURES_TO_SHOW)
)
perm_df.to_csv(OUT_DIR / "permutation_importance.csv", index=False)

plt.figure()
perm_df[::-1].plot(
    x="feature", y="importance_mean", kind="barh", xerr=perm_df[::-1]["importance_std"], legend=False
)
plt.title("Permutation Importance (global, scoring=F1)")
plt.tight_layout()
plt.savefig(OUT_DIR / "permutation_importance.png", dpi=200)
plt.close()


NameError: name 'pipe' is not defined

In [12]:
# ------------------------------------------------------------
# 2) SHAP EXPLANATIONS (global + local)
# ------------------------------------------------------------
print("Computing SHAP values…")
import shap

# Choose best explainer: TreeExplainer for tree models; KernelExplainer otherwise
if any(s in estimator.__class__.__name__.lower() for s in ["xgb", "lgbm", "forest", "tree", "gbm", "boost"]):
    explainer = shap.TreeExplainer(estimator)
    # use transformed data if available (tree uses numeric input)
    data_for_shap = X_trans
    feature_names_for_shap = feature_names_trans
else:
    # Model-agnostic; use small background sample to speed up
    background = X_test.sample(min(300, len(X_test)), random_state=RANDOM_STATE)
    explainer = shap.KernelExplainer(lambda x: pipe.predict_proba(pd.DataFrame(x, columns=raw_feature_names))[:, 1],
                                     background.values)
    data_for_shap = X_test.values
    feature_names_for_shap = raw_feature_names

# Limit SHAP computation to a subset for speed
subset_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test), size=min(1000, len(X_test)), replace=False)
data_subset = data_for_shap[subset_idx]

shap_values = explainer.shap_values(data_subset)
# Unify shape for binary classification
if isinstance(shap_values, list):  # some explainers return [class0, class1]
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

# Summary plot (global drivers)
plt.figure()
shap.summary_plot(shap_vals, data_subset, feature_names=feature_names_for_shap, show=False)
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_summary.png", dpi=200, bbox_inches="tight")
plt.close()

# Bar plot (mean |SHAP| importance)
plt.figure()
shap.summary_plot(shap_vals, data_subset, feature_names=feature_names_for_shap, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_bar.png", dpi=200, bbox_inches="tight")
plt.close()

# One local explanation (first test row)
ix = 0
sample_row = data_for_shap[ix:ix+1]
pred_prob = pipe.predict_proba(X_test.iloc[ix:ix+1])[0, 1]
local_shap = explainer.shap_values(sample_row)
local_vals = local_shap[1] if isinstance(local_shap, list) else local_shap

plt.figure()
shap.force_plot(
    base_value=explainer.expected_value[1] if hasattr(explainer, "expected_value") and isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
    shap_values=local_vals,
    features=sample_row,
    feature_names=feature_names_for_shap,
    matplotlib=True,
    show=False
)
plt.title(f"Local SHAP for Test Row #{ix} (prob={pred_prob:.3f})")
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_local_force.png", dpi=200, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 3) LIME LOCAL EXPLANATION (optional but useful)
# ------------------------------------------------------------
print("Computing LIME explanation for one sample…")
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data=np.array(X_test),
    feature_names=raw_feature_names,
    class_names=["no_default", "default"],
    discretize_continuous=True,
    mode="classification",
    random_state=RANDOM_STATE
)

def predict_fn(X):
    # X is numpy array -> wrap into DataFrame with raw feature names
    return pipe.predict_proba(pd.DataFrame(X, columns=raw_feature_names))

lime_exp = lime_explainer.explain_instance(
    data_row=np.array(X_test.iloc[ix]),
    predict_fn=predict_fn,
    num_features=N_LIME_FEATURES
)

lime_png = OUT_DIR / "lime_local.png"
lime_exp.save_to_file(OUT_DIR / "lime_local.html")  # interactive HTML
try:
    # also save a quick static figure
    fig = lime_exp.as_pyplot_figure()
    fig.tight_layout()
    fig.savefig(lime_png, dpi=200)
    plt.close(fig)
except Exception:
    pass

# ------------------------------------------------------------
# 4) SUMMARIZE RISK INSIGHTS (auto text)
# ------------------------------------------------------------
print("Summarizing risk insights…")
# Global drivers from mean |SHAP|
abs_mean = np.abs(shap_vals).mean(axis=0)
shap_global = (
    pd.DataFrame({"feature": feature_names_for_shap, "mean_abs_shap": abs_mean})
    .sort_values("mean_abs_shap", ascending=False)
    .head(N_FEATURES_TO_SHOW)
)

# Heuristic direction: correlation between feature value and SHAP value
directions = []
for i, feat in enumerate(shap_global["feature"]):
    j = feature_names_for_shap.index(feat)
    corr = np.corrcoef(data_subset[:, j].astype(float), shap_vals[:, j].astype(float))[0, 1]
    if np.isnan(corr):
        dir_text = "direction unclear"
    elif corr > 0.1:
        dir_text = "↑ value tends to ↑ default risk"
    elif corr < -0.1:
        dir_text = "↑ value tends to ↓ default risk"
    else:
        dir_text = "weak/neutral effect"
    directions.append(dir_text)

shap_global["direction"] = directions
shap_global.to_csv(OUT_DIR / "risk_insights_top_features.csv", index=False)

insights = {
    "generated_at": datetime.utcnow().isoformat() + "Z",
    "model": type(estimator).__name__,
    "threshold": 0.5,
    "metrics": {
        "accuracy": report["accuracy"],
        "f1_default": report["1"]["f1-score"],
        "recall_default": report["1"]["recall"],
        "precision_default": report["1"]["precision"]
    },
    "top_global_risk_drivers": shap_global.to_dict(orient="records"),
    "artifacts": {
        "classification_report": str(OUT_DIR / "classification_report.csv"),
        "feature_importance_tree": str(OUT_DIR / "feature_importance_tree.png") if tree_importance_df is not None else None,
        "permutation_importance": str(OUT_DIR / "permutation_importance.png"),
        "shap_summary": str(OUT_DIR / "shap_summary.png"),
        "shap_bar": str(OUT_DIR / "shap_bar.png"),
        "shap_local_force": str(OUT_DIR / "shap_local_force.png"),
        "lime_local_html": str(OUT_DIR / "lime_local.html")
    }
}
with open(OUT_DIR / "risk_insights_summary.json", "w") as f:
    json.dump(insights, f, indent=2)

print("\n✔ Saved artifacts in:", OUT_DIR.resolve())
for k, v in insights["artifacts"].items():
    print(f"- {k}: {v}")


Computing SHAP values…


ModuleNotFoundError: No module named 'shap'